# WNBA Game Prediction, Version 2: An Honest Re-test

Version 1 (`predict.ipynb`) predicted 2023 WNBA home wins with about **70%** accuracy. This notebook checks that number and rebuilds the model so it only uses information a person could have known **before** each game.

**What this notebook shows**
1. Version 1 used end-of-season stats to predict games from that same season, so its 70% was inflated.
2. A fair re-test, run through the season in order, with each game predicted from earlier games only.
3. Every honest approach lands between about **63% and 69%**, and a simple Elo rating does as well as the full model.

In [1]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, brier_score_loss
np.random.seed(0)

## Part 1: Checking Version 1

In [2]:
schedule = pd.read_csv("reg_season.csv").iloc[:, 1:-2]
adv = pd.read_csv("advanced_stats.csv").dropna(axis=1, how="all").iloc[:, 1:-1]
df = pd.merge(schedule, adv, left_on="Visitor/Neutral", right_on="Team")
df = pd.merge(df, adv, left_on="Home/Neutral", right_on="Team").drop(["Team_x", "Team_y"], axis=1)
df["Home_Winner"] = (df["PTS.1"] > df["PTS"]).astype(int)
print("Games:", len(df), "| Home teams won:", round(df.Home_Winner.mean(), 3))

Games: 240 | Home teams won: 0.521


Version 1's predictors come from `advanced_stats.csv`, which holds each team's **final** 2023 numbers, including total wins and losses. Those totals already contain the result of every game being predicted. A quick way to see the problem: simply picking the team with the better final record is right more often than Version 1.

In [3]:
print("Better final record wins:", round(((df.W_y > df.W_x).astype(int) == df.Home_Winner).mean(), 3))

Better final record wins: 0.729


In [4]:
# Version 1 again: predictors chosen on ALL games, then 1,000 random splits
cols = [c for c in df.columns if c not in ["Home/Neutral", "Visitor/Neutral", "PTS", "PTS.1", "Home_Winner"]]
X = pd.DataFrame(MinMaxScaler().fit_transform(df[cols]), columns=cols)
sfs = SequentialFeatureSelector(RidgeClassifier(), n_features_to_select=10, direction="backward").fit(X, df.Home_Winner)
pred_v1 = list(X.columns[sfs.get_support()])
acc = []
for seed in range(1000):
    Xtr, Xte, ytr, yte = train_test_split(X[pred_v1], df.Home_Winner, test_size=0.2, random_state=seed)
    acc.append(accuracy_score(yte, LogisticRegression().fit(Xtr, ytr).predict(Xte)))
acc = np.array(acc)
print("Predictors:", pred_v1)
print(f"Mean accuracy {acc.mean():.3f}; middle 90% of runs {np.percentile(acc,5):.3f} to {np.percentile(acc,95):.3f}")

Predictors: ['Age_x', 'L_x', 'TS%_x', 'Age_y', 'L_y', 'DRtg_y', 'eFG%_y', 'TOV%_y', 'FT/FGA_y', 'eFG%.1_y']
Mean accuracy 0.698; middle 90% of runs 0.603 to 0.792


In [5]:
# Same idea, but predictors are chosen from the training games only (5 runs; slow)
acc_in = []
for seed in range(5):
    tr, te = train_test_split(df.index, test_size=0.2, random_state=seed)
    sc = MinMaxScaler().fit(df.loc[tr, cols])
    Xtr = pd.DataFrame(sc.transform(df.loc[tr, cols]), columns=cols); Xte = pd.DataFrame(sc.transform(df.loc[te, cols]), columns=cols)
    f = SequentialFeatureSelector(RidgeClassifier(), n_features_to_select=10, direction="backward").fit(Xtr, df.loc[tr, "Home_Winner"])
    p = list(Xtr.columns[f.get_support()])
    acc_in.append(accuracy_score(df.loc[te, "Home_Winner"], LogisticRegression().fit(Xtr[p], df.loc[tr, "Home_Winner"]).predict(Xte[p])))
print("Mean accuracy with selection inside the split:", round(np.mean(acc_in), 3))

Mean accuracy with selection inside the split: 0.688


## Part 2: Pre-game features only

For each game, in date order, I build features from **earlier games only**:
- `d_wp`: difference in win percentage so far (lightly smoothed)
- `d_mov`: difference in average point margin so far
- `d_l5`: difference in average margin over each team's last 5 games
- `d_rest`: difference in days of rest (capped at 7)
- `d_elo`: difference in Elo rating (K = 20, margin-adjusted, 50-point home edge)

In [6]:
s = pd.read_csv("reg_season.csv").rename(columns={"Visitor/Neutral": "away", "PTS": "away_pts", "Home/Neutral": "home", "PTS.1": "home_pts"})
s["date"] = pd.to_datetime(s["Date"], format="%a %b %d %Y")
s = s.sort_values("date", kind="stable").reset_index(drop=True)
s["home_win"] = (s.home_pts > s.away_pts).astype(int)

rec = {t: dict(w=0, l=0, margins=[], last=None, elo=1500.0) for t in set(s.home) | set(s.away)}
K, HFA = 20, 50
wp = lambda r: (r["w"] + 1) / (r["w"] + r["l"] + 2)
mov = lambda r: np.mean(r["margins"]) if r["margins"] else 0.0
l5 = lambda r: np.mean(r["margins"][-5:]) if r["margins"] else 0.0
rows = []
for i, g in s.iterrows():
    h, a = rec[g.home], rec[g.away]
    rest = lambda r: min((g.date - r["last"]).days, 7) if r["last"] is not None else 7
    elo_p = 1 / (1 + 10 ** (-(h["elo"] + HFA - a["elo"]) / 400))
    rows.append(dict(i=i, date=g.date, home=g.home, away=g.away, y=g.home_win, gp_min=min(h["w"] + h["l"], a["w"] + a["l"]),
                     d_wp=wp(h) - wp(a), d_mov=mov(h) - mov(a), d_l5=l5(h) - l5(a), d_rest=rest(h) - rest(a), d_elo=h["elo"] - a["elo"], elo_p=elo_p))
    m = g.home_pts - g.away_pts                      # update AFTER recording the pre-game features
    h["margins"].append(m); a["margins"].append(-m)
    if m > 0: h["w"] += 1; a["l"] += 1
    else: a["w"] += 1; h["l"] += 1
    h["last"] = a["last"] = g.date
    d = K * np.log(abs(m) + 1) * ((m > 0) - elo_p); h["elo"] += d; a["elo"] -= d
F = pd.DataFrame(rows)
F.head()

,i,date,home,away,y,gp_min,d_wp,d_mov,d_l5,d_rest,d_elo,elo_p
0,0,2023-05-19,Indiana Fever,Connecticut Sun,0,0,0.0,0.0,0.0,0,0.0,0.571463
1,1,2023-05-19,Los Angeles Sparks,Phoenix Mercury,1,0,0.0,0.0,0.0,0,0.0,0.571463
2,2,2023-05-19,Minnesota Lynx,Chicago Sky,0,0,0.0,0.0,0.0,0,0.0,0.571463
3,3,2023-05-19,Washington Mystics,New York Liberty,1,0,0.0,0.0,0.0,0,0.0,0.571463
4,4,2023-05-20,Dallas Wings,Atlanta Dream,1,0,0.0,0.0,0.0,0,0.0,0.571463


## Part 3: Walk-forward test

Games 81–240 (late June to September) are the test set. Each one is predicted by a model trained only on games played **before that date** (where both teams had played at least 5 games). No game is ever used to predict itself.

In [7]:
ev = F[F.i >= 80].copy()
def walk(feats, C=1.0):
    probs = []
    for _, g in ev.iterrows():
        tr = F[(F.date < g.date) & (F.gp_min >= 5)]
        m = make_pipeline(StandardScaler(), LogisticRegression(C=C)).fit(tr[feats], tr.y)
        probs.append(m.predict_proba(pd.DataFrame([g[feats]]))[0, 1])
    return np.array(probs)
acc = lambda pred: (np.asarray(pred) == ev.y.values).mean()
results = {
    "Always pick the home team": acc(np.ones(len(ev))),
    "Better record so far": acc((ev.d_wp > 0).astype(int)),
    "Better point margin so far": acc((ev.d_mov > 0).astype(int)),
    "Elo favorite": acc((ev.elo_p >= 0.5).astype(int)),
}
for name, feats in [("Model: record + margin", ["d_wp", "d_mov"]), ("Model: all 5 features", ["d_wp", "d_mov", "d_l5", "d_rest", "d_elo"])]:
    p = walk(feats); results[name] = acc((p >= 0.5).astype(int))
    if name.endswith("5 features"): p_full = p
table = pd.Series(results).round(3).sort_values(ascending=False)
print("Test games:", len(ev)); table

Test games: 160


Model: record + margin        0.688
Elo favorite                  0.688
Better point margin so far    0.669
Better record so far          0.662
Model: all 5 features         0.625
Always pick the home team     0.538
dtype: float64

In [8]:
# How much could these numbers move by chance? Bootstrap the 160 test games.
rng = np.random.RandomState(1)
hits = ((p_full >= 0.5).astype(int) == ev.y.values)
bs = [hits[rng.randint(0, len(hits), len(hits))].mean() for _ in range(2000)]
print(f"Full model accuracy {hits.mean():.3f}; 90% interval {np.percentile(bs,5):.3f} to {np.percentile(bs,95):.3f}")

Full model accuracy 0.625; 90% interval 0.562 to 0.688


With only 160 test games, a few points of difference is within noise. Every honest approach lands between about 63% and 69%, and adding more features did not help.

## Part 4: The 2023 Finals

In [9]:
final = make_pipeline(StandardScaler(), LogisticRegression()).fit(F[F.gp_min >= 5][["d_wp", "d_mov"]], F[F.gp_min >= 5].y)
def state(t): r = rec[t]; return wp(r), np.mean(r["margins"])
A, L = state("Las Vegas Aces"), state("New York Liberty")
for home, hs, aw in (("Las Vegas Aces", A, L), ("New York Liberty", L, A)):
    x = pd.DataFrame([dict(d_wp=hs[0] - aw[0], d_mov=hs[1] - aw[1])])
    print(f"{home} at home: home win probability {final.predict_proba(x)[0,1]:.2f}")

Las Vegas Aces at home: home win probability 0.59
New York Liberty at home: home win probability 0.45


The model favors the Aces wherever the game is played. The Aces won the 2023 Finals 3–1, so it would have called 3 of the 4 games, missing only Game 3 in New York.

## Takeaways
- **Check for leakage first.** Season-end stats made Version 1 look better than it was.
- **Always compare with a simple baseline.** A one-line rule matched or beat the model.
- **Report uncertainty.** With one season of data, differences of a few points are noise.
- **Next steps:** add prior seasons, player availability and injuries, and test on 2024.